# Patent Landscaping — Snorkel vs MAS (Colab GPU side)

**Split of work**
- **Local `.venv` (done beforehand):** data pipeline + dedup + negative pool + **MAS labeling (10 keys)**. `DataSet/mas/mas_ranked_scores.csv` is committed, so it arrives with the clone.
- **This Colab notebook (GPU):** Snorkel labeling + **SciBERT fine-tune & eval for BOTH arms**, train/val curves, threshold calibration + ROC/PR.

Runtime → change runtime type → **GPU (T4)**. SciBERT fine-tune ≈ 20–40 min/arm.

**Persistence:** trained models live in `outputs/` on Colab's *temporary* disk (wiped on disconnect). Cell **5c saves them to Google Drive**; a future session can **restore (cell R)** and skip the 1-hour training — just re-run the fast cells 1–3 first.

In [ ]:
# 1) clone + install
REPO = 'https://github.com/PassionChicken-Leesuin/Patent_Landscaping_Final.git'
import os
if not os.path.exists('Patent_Landscaping_Final'):
    !git clone $REPO
%cd Patent_Landscaping_Final
!git pull        # get latest fixes
!pip -q install snorkel transformers datasets scikit-learn accelerate

In [ ]:
# 2) (re)build processed datasets from raw (dedup + negative pool)
!python -m scripts.build_dataset

In [ ]:
# 3) SNORKEL arm — label the candidate pool with LabelModel
!python -m scripts.run_snorkel

MAS labels (`DataSet/mas/mas_ranked_scores.csv`) are already in the repo from the local 10-key run — nothing to upload.

> **Already trained before and saved to Drive?** Skip cells 4–5 and jump to **cell R (restore)**, then run calibration/compare.

In [ ]:
# 4) downstream: SNORKEL arm (assemble -> fine-tune SciBERT -> eval)
!python -m scripts.run_downstream --arm snorkel --epochs 4

In [ ]:
# 4b) training/validation curves (baseline Fig 3/4 style)
from src.downstream.plots import plot_history
plot_history('snorkel')

In [ ]:
# 5) downstream: MAS arm
!python -m scripts.run_downstream --arm mas --epochs 4

In [ ]:
# 5b) training/validation curves for the MAS arm
from src.downstream.plots import plot_history
plot_history('mas')

### 5c) SAVE trained models to Google Drive (run now, while the session is alive)
Persists `outputs/` (both SciBERT checkpoints + history + metrics + curves) so you never have to retrain.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/patent_landscaping_outputs'
import os
os.makedirs(DRIVE, exist_ok=True)
!cp -r outputs/* "{DRIVE}/"
print('saved outputs -> ', DRIVE)
!du -sh "{DRIVE}"/* 2>/dev/null

In [ ]:
# 6) compare final metrics on the gold eval set (threshold 0.5)
import json
for arm in ['snorkel', 'mas']:
    r = json.load(open(f'outputs/metrics_{arm}.json'))
    print(f"{arm:8s}  macroF1={r['macro_f1']:.4f}  AUC={r['auc']:.4f}  "
          f"P={r['precision']:.4f}  R={r['recall']:.4f}  acc={r['accuracy']:.4f}")
    print('          by level:', {k: v for k, v in r['by_expansion_level'].items()})

## 7) Threshold calibration + ROC/PR diagnostics
At threshold 0.5 both models predict almost all-negative (recall ~6–12%) because the OOD negatives are too easy — a threshold problem, not a capability one (AUC ≈ 0.71–0.75). This tunes the threshold on the **validation** split (never the gold set) and re-reports on gold, plus ROC/PR/score-distribution plots.

In [ ]:
!python -m scripts.calibrate_eval

---
## Cell R) RESTORE models from Drive (use in a FRESH session — no retraining)
Run cells **1, 2, 3** first (clone+install, build_dataset, run_snorkel — all fast), then this cell, then jump straight to **6 / 7**.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/patent_landscaping_outputs'
import os
os.makedirs('outputs', exist_ok=True)
!cp -r "{DRIVE}/"* outputs/
print('restored outputs from', DRIVE)
!ls outputs